# Retrieval Playground — coding workshop (advanced)

About 20–25 minutes, individual. No GPU and no downloads needed: any Colab runtime works.

Sparse retrieval ranks documents by word overlap; dense retrieval ranks them by embedding
similarity. Here you build a tiny version of both by hand, over five toy documents. Notebook
3's Part 2 then runs the same comparison over the 26-document course corpus.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `ExerciseGaps: this cell still has gaps marked ___`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 3 above).
fill_in_solutions = False  # @param {type:"boolean"}

import io
import re
import tokenize

_solutions = {
    1: "TOY_DOCS = {\n    \"doc_carbon\": \"training a model produces carbon emissions measured once\",\n    \"doc_energy\": \"running a model costs energy every single time it answers a query\",\n    \"doc_water\": \"data centers consume water for cooling as well as electricity\",\n    \"doc_retrieval\": \"a retriever finds relevant documents before the model reads them\",\n    \"doc_quantize\": \"quantization stores numbers with fewer bits to save memory\",\n}\n\ndef word_overlap_score(query, doc_text):\n    query_words = set(query.lower().split())\n    doc_words = set(doc_text.lower().split())\n    return len(query_words & doc_words)\n\n\ndef rank_by_overlap(query):\n    scores = {name: word_overlap_score(query, text) for name, text in TOY_DOCS.items()}\n    for name, score in sorted(scores.items(), key=lambda x: -x[1]):\n        shared = set(query.lower().split()) & set(TOY_DOCS[name].split())\n        print(f\"{name:14s} {score} shared: {sorted(shared)}\")\n\n\nQUERY_A = \"how much energy does running a model use\"\nrank_by_overlap(QUERY_A)",
    2: "import math\n\ndef idf(word, docs):\n    doc_count = sum(1 for text in docs.values() if word in text.lower().split())\n    return math.log(len(docs) / doc_count)\n\ndef tf(word, doc_text):\n    return doc_text.lower().split().count(word)\n\nfor word in [\"model\", \"energy\"]:\n    print(f\"idf('{word}') = {idf(word, TOY_DOCS):.3f}\")\n\nprint(\"\\nInside doc_energy, both words appear exactly once (tf = 1 for both):\")\nfor word in [\"model\", \"energy\"]:\n    score = tf(word, TOY_DOCS[\"doc_energy\"]) * idf(word, TOY_DOCS)\n    print(f\"  tf-idf('{word}', doc_energy) = {score:.3f}\")\n\n\ndef tfidf_score(query, doc_text, docs=TOY_DOCS):\n    total = 0.0\n    for word in set(query.lower().split()):\n        if any(word in text.lower().split() for text in docs.values()):\n            total += tf(word, doc_text) * idf(word, docs)\n    return total\n\n\nfor query in [QUERY_A, QUERY_B]:\n    print(f\"\\n{query}\")\n    for name, text in sorted(TOY_DOCS.items(), key=lambda kv: -tfidf_score(query, kv[1])):\n        print(f\"  {name:14s} {tfidf_score(query, text):.3f}\")",
    3: "def avg_doc_length(docs):\n    return sum(len(text.lower().split()) for text in docs.values()) / len(docs)\n\n\ndef bm25_idf(word, docs):\n    n_t = sum(1 for text in docs.values() if word in text.lower().split())\n    return math.log(1 + (len(docs) - n_t + 0.5) / (n_t + 0.5))\n\n\ndef bm25_score(query, doc_name, docs, k1=1.2, b=0.75):\n    words = docs[doc_name].lower().split()\n    score = 0.0\n    for t in set(query.lower().split()):\n        tf = words.count(t)\n        if tf == 0:\n            continue\n        length_norm = 1 - b + b * len(words) / avg_doc_length(docs)\n        score += bm25_idf(t, docs) * tf * (k1 + 1) / (tf + k1 * length_norm)\n    return score\n\nprint(f\"Average document length: {avg_doc_length(TOY_DOCS):.1f} words\\n\")\nfor query in [QUERY_A, QUERY_B]:\n    print(f\"Query: {query}\")\n    scores = {name: bm25_score(query, name, TOY_DOCS) for name in TOY_DOCS}\n    for name, score in sorted(scores.items(), key=lambda x: -x[1]):\n        print(f\"  {name:14s} {score:.3f}\")\n    print()",
    4: "import numpy as np\n\nTOY_WORD_VECTORS = {\n    \"energy\":      np.array([4, 2]),\n    \"electricity\": np.array([4, 3]),\n    \"carbon\":      np.array([3, 4]),\n    \"water\":       np.array([1, 4]),\n    \"retriever\":   np.array([0, 5]),\n    \"memory\":      np.array([2, -3]),\n}\n\ndef cosine_sim(v1, v2):\n    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))\n\ndef text_vector(text):\n    words = [w for w in text.lower().split() if w in TOY_WORD_VECTORS]\n    return np.mean([TOY_WORD_VECTORS[w] for w in words], axis=0)\n\nTOY_DOC_VECTORS = {name: text_vector(text) for name, text in TOY_DOCS.items()}\nfor name, vec in TOY_DOC_VECTORS.items():\n    print(f\"{name:14s} {vec}\")",
    5: "query_b_vector = text_vector(QUERY_B)\nprint(\"QUERY_B vector:\", query_b_vector, \"\\n\")\n\ndense_scores_b = {name: cosine_sim(query_b_vector, vec) for name, vec in TOY_DOC_VECTORS.items()}\nfor name, score in sorted(dense_scores_b.items(), key=lambda x: -x[1]):\n    print(f\"{name:14s} cosine similarity {score:.3f}\")",
    6: "from collections import defaultdict\n\n\ndef build_inverted_index(docs):\n    index = defaultdict(dict)  # word -> {doc_name: tf}\n    for name, text in docs.items():\n        for word in text.lower().split():\n            index[word][name] = index[word].get(name, 0) + 1\n    return index\n\n\ndef bm25_with_index(query, index, docs, k1=1.2, b=0.75):\n    avgdl = avg_doc_length(docs)\n    scores = defaultdict(float)\n    n_postings = 0\n    for t in set(query.lower().split()):\n        idf_t = bm25_idf(t, docs)\n        for name, tf in index.get(t, {}).items():\n            n_postings += 1\n            length_norm = 1 - b + b * len(docs[name].split()) / avgdl\n            scores[name] += idf_t * tf * (k1 + 1) / (tf + k1 * length_norm)\n    return dict(scores), n_postings\n\nindex = build_inverted_index(TOY_DOCS)\nprint(\"Postings for 'model':\", index[\"model\"])\n\nEMBEDDING_DIM = 384  # paraphrase-multilingual-MiniLM-L12-v2, the encoder in Notebook 3\nfor query in [QUERY_A, QUERY_B]:\n    scores, n_postings = bm25_with_index(query, index, TOY_DOCS)\n    print(f\"\\nQuery: {query}\")\n    print(f\"  sparse: {n_postings} postings visited, {len(scores)} of {len(TOY_DOCS)} documents scored\")\n    print(f\"  dense:  {len(TOY_DOCS) * EMBEDDING_DIM} multiply-adds (every document, every dimension)\")",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


class ExerciseGaps(Exception):
    """A cell still has gaps marked ___."""


def _has_gap(lines):
    """True if `___` appears in the cell as code (not in a string or a comment)."""
    code = "".join(l for l in lines if not l.lstrip().startswith(("%", "!")))
    try:
        return any(t.type == tokenize.NAME and t.string == "___"
                   for t in tokenize.generate_tokens(io.StringIO(code).readline))
    except (tokenize.TokenError, SyntaxError):
        return False


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution.
    A cell that still has gaps ___ stops with a message. (In IPython, ___ is a built-in name, the
    third-to-last output, so without this check the cell would run on with a wrong value.)"""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if m and int(m.group(1)) in _fill_in:
        n = int(m.group(1))
        return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)
    if _has_gap(lines):
        return ['raise ExerciseGaps("this cell still has gaps marked ___: fill them in and run it again, '
                'or tick fill_in_solutions in the first code cell and run that cell")\n']
    return lines


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Part A — Sparse retrieval, by hand

Five toy documents on topics from this course. No library and no TF-IDF yet: count how
many query words appear in each document, and rank by that count. This is the crudest
possible sparse retrieval, which makes both the mechanism and its failure mode easy to see.

### ✏️ Exercise 1 — Word-overlap scoring

Complete `word_overlap_score`: the number of distinct words that the query and the
document share, ignoring case.

<details><summary><b>Hints</b> (click to open)</summary>

- `set(...)` keeps each word once.
- `a & b` is the set of elements in both `a` and `b`.

</details>

In [ ]:
# Exercise 1
TOY_DOCS = {
    "doc_carbon": "training a model produces carbon emissions measured once",
    "doc_energy": "running a model costs energy every single time it answers a query",
    "doc_water": "data centers consume water for cooling as well as electricity",
    "doc_retrieval": "a retriever finds relevant documents before the model reads them",
    "doc_quantize": "quantization stores numbers with fewer bits to save memory",
}

def word_overlap_score(query, doc_text):
    query_words = ___  # TODO: the distinct lowercase words of the query
    doc_words = ___  # TODO: the same for the document
    return ___  # TODO: how many words they share


def rank_by_overlap(query):
    scores = {name: word_overlap_score(query, text) for name, text in TOY_DOCS.items()}
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        shared = set(query.lower().split()) & set(TOY_DOCS[name].split())
        print(f"{name:14s} {score} shared: {sorted(shared)}")


QUERY_A = "how much energy does running a model use"
rank_by_overlap(QUERY_A)

In [ ]:
# Check for Exercise 1. Run this cell as is.
assert word_overlap_score("Energy model", "a model uses energy") == 2
assert word_overlap_score("a a a", "a") == 1, "count each shared word once"
assert word_overlap_score("water", "energy") == 0
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
TOY_DOCS = {
    "doc_carbon": "training a model produces carbon emissions measured once",
    "doc_energy": "running a model costs energy every single time it answers a query",
    "doc_water": "data centers consume water for cooling as well as electricity",
    "doc_retrieval": "a retriever finds relevant documents before the model reads them",
    "doc_quantize": "quantization stores numbers with fewer bits to save memory",
}

def word_overlap_score(query, doc_text):
    query_words = set(query.lower().split())
    doc_words = set(doc_text.lower().split())
    return len(query_words & doc_words)


def rank_by_overlap(query):
    scores = {name: word_overlap_score(query, text) for name, text in TOY_DOCS.items()}
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        shared = set(query.lower().split()) & set(TOY_DOCS[name].split())
        print(f"{name:14s} {score} shared: {sorted(shared)}")


QUERY_A = "how much energy does running a model use"
rank_by_overlap(QUERY_A)
```

</details>

## Try it yourself: a paraphrased query

`QUERY_B` asks essentially the same question as `QUERY_A` in different words, with no
`"energy"` and no `"model"`. Run the cell and see what word-overlap scoring does with it.

In [ ]:
QUERY_B = "what does it cost in electricity to get an answer"
rank_by_overlap(QUERY_B)

**Reflection:** Compare the two rankings, for `QUERY_A` and `QUERY_B`. `QUERY_B` asks
essentially the same thing as `QUERY_A` in different words. Did word-overlap scoring still
find `doc_energy`? Look at the `shared` words: which of them tell you anything about the
topic? What would need to be true about a query and a document for this scoring method to
work?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

No. For `QUERY_A`, `doc_energy` wins clearly with 4 shared words. For `QUERY_B`, it ties with `doc_water` and `doc_quantize` at 1 shared word each, and comes first only because of the order of the dictionary.

The shared words are `it` (with `doc_energy`), `to` (with `doc_quantize`) and `electricity` (with `doc_water`). `it` and `to` say nothing about the topic. `electricity` is the only topical word, and it points to the wrong document: `doc_water` mentions electricity only in passing. Even for `QUERY_A`, part of the score comes from `a`, which is also why `doc_carbon` and `doc_retrieval` score 2.

For word overlap to work, the query has to use the same words as the document, in the same form, and the shared words have to be the ones that carry the topic. A paraphrase breaks the first condition, and filler words break the second.

</details>

## Part A, continued — weighting by rarity (TF-IDF)

Word-overlap scoring treats every match the same: one shared word is one point, whether the
word is `"a"` or `"energy"`. TF-IDF scores a word higher the rarer it is across the
collection, so a distinctive word like `energy` counts for more than a common one like
`model`.

Two ingredients:
- **tf** (term frequency): how many times does the word appear in *this* document?
- **idf** (inverse document frequency): how rare is the word across *all* the documents?
  `idf(word) = log(number of documents / documents containing the word)`

Score = tf × idf. The cell below computes both for two words in `doc_energy`: `model`
(in 3 of the 5 documents) and `energy` (in only 1).

### ✏️ Exercise 2 — TF-IDF by hand

Three parts:

1. `idf(word, docs)` = log(number of documents / number of documents that contain `word`).
2. `tf(word, doc_text)` = how many times `word` occurs in the document.
3. At the end of the cell, `tfidf_score(query, doc_text)` adds up tf × idf over the
   distinct query words. It skips words that occur in no document, since their idf would
   divide by zero. The cell then ranks the five documents for `QUERY_A` and `QUERY_B`.

<details><summary><b>Hints</b> (click to open)</summary>

- `math.log` is the natural logarithm.
- `list.count(x)` counts how many times `x` occurs in a list.
- For each query word: `tf(word, doc_text) * idf(word, docs)`.

</details>

In [ ]:
# Exercise 2
import math

def idf(word, docs):
    doc_count = ___  # TODO: number of documents whose words include word
    return ___

def tf(word, doc_text):
    return ___  # TODO: occurrences of word in doc_text

for word in ["model", "energy"]:
    print(f"idf('{word}') = {idf(word, TOY_DOCS):.3f}")

print("\nInside doc_energy, both words appear exactly once (tf = 1 for both):")
for word in ["model", "energy"]:
    score = tf(word, TOY_DOCS["doc_energy"]) * idf(word, TOY_DOCS)
    print(f"  tf-idf('{word}', doc_energy) = {score:.3f}")


def tfidf_score(query, doc_text, docs=TOY_DOCS):
    # TODO: sum of tf * idf over the distinct query words that occur in some document
    ___


for query in [QUERY_A, QUERY_B]:
    print(f"\n{query}")
    for name, text in sorted(TOY_DOCS.items(), key=lambda kv: -tfidf_score(query, kv[1])):
        print(f"  {name:14s} {tfidf_score(query, text):.3f}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert abs(idf("model", TOY_DOCS) - math.log(5 / 3)) < 1e-9, "'model' occurs in 3 of the 5 documents"
assert abs(idf("energy", TOY_DOCS) - math.log(5)) < 1e-9
assert tf("a", "a b a") == 2
assert max(TOY_DOCS, key=lambda n: tfidf_score(QUERY_A, TOY_DOCS[n])) == "doc_energy"
assert tfidf_score("zzz", TOY_DOCS["doc_energy"]) == 0
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
import math

def idf(word, docs):
    doc_count = sum(1 for text in docs.values() if word in text.lower().split())
    return math.log(len(docs) / doc_count)

def tf(word, doc_text):
    return doc_text.lower().split().count(word)

for word in ["model", "energy"]:
    print(f"idf('{word}') = {idf(word, TOY_DOCS):.3f}")

print("\nInside doc_energy, both words appear exactly once (tf = 1 for both):")
for word in ["model", "energy"]:
    score = tf(word, TOY_DOCS["doc_energy"]) * idf(word, TOY_DOCS)
    print(f"  tf-idf('{word}', doc_energy) = {score:.3f}")


def tfidf_score(query, doc_text, docs=TOY_DOCS):
    total = 0.0
    for word in set(query.lower().split()):
        if any(word in text.lower().split() for text in docs.values()):
            total += tf(word, doc_text) * idf(word, docs)
    return total


for query in [QUERY_A, QUERY_B]:
    print(f"\n{query}")
    for name, text in sorted(TOY_DOCS.items(), key=lambda kv: -tfidf_score(query, kv[1])):
        print(f"  {name:14s} {tfidf_score(query, text):.3f}")
```

</details>

**Reflection:** Why does `energy` score higher than `model`, even though both appear
exactly once in `doc_energy`? Now run `idf("it", TOY_DOCS)` or `idf("to", TOY_DOCS)`
yourself. What do you get, and why might that number be misleading on a collection this
small?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Both words appear once in `doc_energy`, so tf is 1 for both and the score is just the idf. `energy` occurs in only 1 of the 5 documents, so its idf is ln(5/1), about 1.61. `model` occurs in 3 of them, so its idf is ln(5/3), about 0.51. The rarer word says more about which document you want, and it counts more.

`idf("it", TOY_DOCS)` and `idf("to", TOY_DOCS)` both give 1.609, the same as `energy` and the highest value possible here. Each happens to occur in only one of the five short documents. In a real collection, `it` and `to` appear in nearly every document, and their idf would be close to 0. With five documents, "rare" is often an accident. You can see the effect in the TF-IDF ranking for `QUERY_B`: `it`, `to` and `electricity` each give 1.609, so the three documents still tie, and TF-IDF does no better than plain word overlap on this query.

</details>

### ✏️ Exercise 3 — BM25 from its formula (advanced)

TF-IDF keeps growing with every repeat of a word, and long documents collect more
matches just by being long. **BM25** (Robertson & Zaragoza 2009, *The Probabilistic
Relevance Framework: BM25 and Beyond*) fixes both. It is the default ranking function of
search engines such as Lucene and Elasticsearch. For a query q and a document d:

BM25(q, d) = Σ over words t in q of  IDF(t) · tf · (k₁ + 1) / ( tf + k₁ · (1 − b + b · |d| / avgdl) )

- tf: the number of times t occurs in d;
- |d|: the length of d in words; avgdl: the average document length in the collection;
- IDF(t) = ln( 1 + (N − nₜ + 0.5) / (nₜ + 0.5) ), with N documents of which nₜ contain t;
- k₁ = 1.2 limits how much repeats count; b = 0.75 sets how much length matters.

Complete the three functions. As in the cells above, a text's words are `text.lower().split()`,
and each query word counts once.

<details><summary><b>Hints</b> (click to open)</summary>

- `sum(len(text.lower().split()) for text in docs.values()) / len(docs)`.
- `math.log(1 + (len(docs) - n_t + 0.5) / (n_t + 0.5))`.
- Compute the length part first: `1 - b + b * len(words) / avg_doc_length(docs)`.

</details>

In [ ]:
# Exercise 3
def avg_doc_length(docs):
    # TODO: the average number of words per document
    ___


def bm25_idf(word, docs):
    n_t = sum(1 for text in docs.values() if word in text.lower().split())
    # TODO: the BM25 IDF
    ___


def bm25_score(query, doc_name, docs, k1=1.2, b=0.75):
    words = docs[doc_name].lower().split()
    score = 0.0
    for t in set(query.lower().split()):
        tf = words.count(t)
        if tf == 0:
            continue
        # TODO: add this word's BM25 term to score
        ___
    return score

print(f"Average document length: {avg_doc_length(TOY_DOCS):.1f} words\n")
for query in [QUERY_A, QUERY_B]:
    print(f"Query: {query}")
    scores = {name: bm25_score(query, name, TOY_DOCS) for name in TOY_DOCS}
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {name:14s} {score:.3f}")
    print()

In [ ]:
# Check for Exercise 3. Run this cell as is.
_docs = {"d1": "cat sat", "d2": "dog ran", "d3": "cat cat cat cat"}
assert avg_doc_length(_docs) == 8 / 3
assert abs(bm25_idf("dog", _docs) - math.log(1 + 2.5 / 1.5)) < 1e-12
assert abs(bm25_score("dog", "d2", {"d2": "dog ran", "d4": "cow ate"}) - bm25_idf("dog", {"d2": "dog ran", "d4": "cow ate"})) < 1e-12, \
    "tf = 1 in a document of average length: the score is just the IDF"
assert bm25_score("dog", "d1", _docs) == 0
_s = [bm25_score("cat", "d3", {**_docs, "d3": " ".join(["cat"] * n)}, b=0) for n in (1, 2, 4, 50)]
assert _s[0] < _s[1] < _s[2] < _s[3] < _s[0] * (1.2 + 1) * 1.01, "repeats help less and less: at most (k1 + 1) x"
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def avg_doc_length(docs):
    return sum(len(text.lower().split()) for text in docs.values()) / len(docs)


def bm25_idf(word, docs):
    n_t = sum(1 for text in docs.values() if word in text.lower().split())
    return math.log(1 + (len(docs) - n_t + 0.5) / (n_t + 0.5))


def bm25_score(query, doc_name, docs, k1=1.2, b=0.75):
    words = docs[doc_name].lower().split()
    score = 0.0
    for t in set(query.lower().split()):
        tf = words.count(t)
        if tf == 0:
            continue
        length_norm = 1 - b + b * len(words) / avg_doc_length(docs)
        score += bm25_idf(t, docs) * tf * (k1 + 1) / (tf + k1 * length_norm)
    return score

print(f"Average document length: {avg_doc_length(TOY_DOCS):.1f} words\n")
for query in [QUERY_A, QUERY_B]:
    print(f"Query: {query}")
    scores = {name: bm25_score(query, name, TOY_DOCS) for name in TOY_DOCS}
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {name:14s} {score:.3f}")
    print()
```

</details>

## Part B — Dense retrieval, by hand

Dense retrieval compares vectors instead of strings. A real system gets its vectors from a
trained embedding model; here we use hand-picked 2-D toy vectors, the same approach as the
lecture's aunt/uncle/car example, so every number is visible. A document's vector is the
average of the vectors of the words we defined.

### ✏️ Exercise 4 — Cosine similarity and text vectors

Complete `cosine_sim` and `text_vector`. A text's vector is the average of the toy
vectors of its words that have one.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.linalg.norm(v)` is the length of `v`.
- Averaging a list of 2-D vectors element by element: `np.mean(list_of_vectors, axis=0)`.

</details>

In [ ]:
# Exercise 4
import numpy as np

TOY_WORD_VECTORS = {
    "energy":      np.array([4, 2]),
    "electricity": np.array([4, 3]),
    "carbon":      np.array([3, 4]),
    "water":       np.array([1, 4]),
    "retriever":   np.array([0, 5]),
    "memory":      np.array([2, -3]),
}

def cosine_sim(v1, v2):
    return ___  # TODO: cosine similarity of v1 and v2, as a float

def text_vector(text):
    words = [w for w in text.lower().split() if w in TOY_WORD_VECTORS]
    return ___  # TODO: the average of the vectors of the words in `words`

TOY_DOC_VECTORS = {name: text_vector(text) for name, text in TOY_DOCS.items()}
for name, vec in TOY_DOC_VECTORS.items():
    print(f"{name:14s} {vec}")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert abs(cosine_sim(np.array([4, 2]), np.array([3, 3])) - 0.949) < 1e-3, "Session 1 example: cos(aunt, uncle)"
assert np.allclose(text_vector("energy and water"), [2.5, 3.0]), text_vector("energy and water")
assert np.allclose(TOY_DOC_VECTORS["doc_energy"], [4, 2])
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
import numpy as np

TOY_WORD_VECTORS = {
    "energy":      np.array([4, 2]),
    "electricity": np.array([4, 3]),
    "carbon":      np.array([3, 4]),
    "water":       np.array([1, 4]),
    "retriever":   np.array([0, 5]),
    "memory":      np.array([2, -3]),
}

def cosine_sim(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def text_vector(text):
    words = [w for w in text.lower().split() if w in TOY_WORD_VECTORS]
    return np.mean([TOY_WORD_VECTORS[w] for w in words], axis=0)

TOY_DOC_VECTORS = {name: text_vector(text) for name, text in TOY_DOCS.items()}
for name, vec in TOY_DOC_VECTORS.items():
    print(f"{name:14s} {vec}")
```

</details>

## Rank the paraphrased query by cosine similarity

Same `QUERY_B` as in Part A, now embedded with the same averaging. The only word it has a
toy vector for is `"electricity"`, so that is its vector.

### ✏️ Exercise 5 — Rank by cosine similarity

Score every document against the query vector, in a dict from document name to
cosine similarity.

<details><summary><b>Hints</b> (click to open)</summary>

- Loop over `TOY_DOC_VECTORS.items()`: each item is a name and its vector.

</details>

In [ ]:
# Exercise 5
query_b_vector = text_vector(QUERY_B)
print("QUERY_B vector:", query_b_vector, "\n")

dense_scores_b = ___  # TODO: {document name: cosine similarity to query_b_vector}
for name, score in sorted(dense_scores_b.items(), key=lambda x: -x[1]):
    print(f"{name:14s} cosine similarity {score:.3f}")

In [ ]:
# Check for Exercise 5. Run this cell as is.
assert set(dense_scores_b) == set(TOY_DOCS)
assert max(dense_scores_b, key=dense_scores_b.get) == "doc_energy"
assert abs(dense_scores_b["doc_carbon"] - 0.96) < 1e-9
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
query_b_vector = text_vector(QUERY_B)
print("QUERY_B vector:", query_b_vector, "\n")

dense_scores_b = {name: cosine_sim(query_b_vector, vec) for name, vec in TOY_DOC_VECTORS.items()}
for name, score in sorted(dense_scores_b.items(), key=lambda x: -x[1]):
    print(f"{name:14s} cosine similarity {score:.3f}")
```

</details>

**Reflection:** Compare this ranking to `QUERY_B`'s word-overlap ranking in Part A. Did
`doc_energy` do better? What is different about how this score was computed? The toy
vectors were hand-picked so that "electricity" sits close to "energy". Where does that
closeness come from in a trained embedding model? (Think back to the skip-gram lecture.)
Look at how close the top three scores are: what might that mean for a retriever? And
what does `doc_quantize`'s negative score tell you about the angle between the two
vectors?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Yes. In our run, `doc_energy` comes first with 0.984, where word overlap only tied it with two others. The dense score compares vectors, not strings: `electricity` never appears in `doc_energy`, but its vector points in almost the same direction as the vector of `energy`, so the paraphrase is still found.

In a trained embedding model, that closeness comes from context. Skip-gram learns each word's vector by predicting the words around it, so words used in similar sentences, such as *electricity* and *energy*, end up with similar vectors.

The top three scores are within about 0.03 of each other. A small change in the vectors or in the query's wording could change the order, so a retriever that keeps only the first document could easily pick a wrong one. Keeping a few top documents is safer.

The negative score for `doc_quantize` means the angle between the two vectors is a little over 90° (about 93°): the two vectors are almost perpendicular, so in this toy space `memory` has nothing to do with `electricity`.

</details>

### ✏️ Exercise 6 — An inverted index: score only the documents that match (advanced)

Dense retrieval compares the query vector with **every** document vector, in every
dimension. Sparse retrieval can skip most documents: an **inverted index** maps each word
to its **postings**, the documents that contain it and how often. To score a query, visit
only the postings of the query words; every other document scores 0.

Complete `build_inverted_index` and `bm25_with_index`, using `bm25_idf` and the length
normalization from the BM25 exercise. The cell compares the work done with what a dense
scan over 384-dimensional vectors would do.

**Reflection:** with 5 documents the difference is small. In a collection of 10 million
web pages, how many documents contain the word "energy", and how many contain "the"? What
does that mean for the cost of sparse retrieval, and why do search engines remove very
common words from queries?

<details><summary><b>Hints</b> (click to open)</summary>

- `index[word].get(name, 0) + 1` is the new count.
- The BM25 term is the same as in `bm25_score`: `idf_t * tf * (k1 + 1) / (tf + k1 * length_norm)`.
- `scores` is a `defaultdict(float)`, so `scores[name] += ...` works for a new document.

</details>

In [ ]:
# Exercise 6
from collections import defaultdict


def build_inverted_index(docs):
    index = defaultdict(dict)  # word -> {doc_name: tf}
    for name, text in docs.items():
        for word in text.lower().split():
            # TODO: count one more occurrence of word in document name
            ___
    return index


def bm25_with_index(query, index, docs, k1=1.2, b=0.75):
    avgdl = avg_doc_length(docs)
    scores = defaultdict(float)
    n_postings = 0
    for t in set(query.lower().split()):
        idf_t = bm25_idf(t, docs)
        for name, tf in index.get(t, {}).items():
            n_postings += 1
            length_norm = 1 - b + b * len(docs[name].split()) / avgdl
            # TODO: add this word's BM25 term to the document's score
            ___
    return dict(scores), n_postings

index = build_inverted_index(TOY_DOCS)
print("Postings for 'model':", index["model"])

EMBEDDING_DIM = 384  # paraphrase-multilingual-MiniLM-L12-v2, the encoder in Notebook 3
for query in [QUERY_A, QUERY_B]:
    scores, n_postings = bm25_with_index(query, index, TOY_DOCS)
    print(f"\nQuery: {query}")
    print(f"  sparse: {n_postings} postings visited, {len(scores)} of {len(TOY_DOCS)} documents scored")
    print(f"  dense:  {len(TOY_DOCS) * EMBEDDING_DIM} multiply-adds (every document, every dimension)")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert build_inverted_index({"d1": "a b a", "d2": "b"}) == {"a": {"d1": 2}, "b": {"d1": 1, "d2": 1}}
for _q in [QUERY_A, QUERY_B]:
    _scores, _ = bm25_with_index(_q, index, TOY_DOCS)
    for _name in TOY_DOCS:
        assert abs(_scores.get(_name, 0.0) - bm25_score(_q, _name, TOY_DOCS)) < 1e-9, \
            f"{_name}: the indexed score should equal bm25_score"
_, _n = bm25_with_index(QUERY_A, index, TOY_DOCS)
assert _n == sum(len(index.get(t, {})) for t in set(QUERY_A.lower().split()))
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
from collections import defaultdict


def build_inverted_index(docs):
    index = defaultdict(dict)  # word -> {doc_name: tf}
    for name, text in docs.items():
        for word in text.lower().split():
            index[word][name] = index[word].get(name, 0) + 1
    return index


def bm25_with_index(query, index, docs, k1=1.2, b=0.75):
    avgdl = avg_doc_length(docs)
    scores = defaultdict(float)
    n_postings = 0
    for t in set(query.lower().split()):
        idf_t = bm25_idf(t, docs)
        for name, tf in index.get(t, {}).items():
            n_postings += 1
            length_norm = 1 - b + b * len(docs[name].split()) / avgdl
            scores[name] += idf_t * tf * (k1 + 1) / (tf + k1 * length_norm)
    return dict(scores), n_postings

index = build_inverted_index(TOY_DOCS)
print("Postings for 'model':", index["model"])

EMBEDDING_DIM = 384  # paraphrase-multilingual-MiniLM-L12-v2, the encoder in Notebook 3
for query in [QUERY_A, QUERY_B]:
    scores, n_postings = bm25_with_index(query, index, TOY_DOCS)
    print(f"\nQuery: {query}")
    print(f"  sparse: {n_postings} postings visited, {len(scores)} of {len(TOY_DOCS)} documents scored")
    print(f"  dense:  {len(TOY_DOCS) * EMBEDDING_DIM} multiply-adds (every document, every dimension)")
```

</details>

## Code literacy check

Before you move on, pick at least two code cells above that you ran and read today, and
add `#` comments explaining in your own words what each meaningful line or block does and
*why* it's there, so that a classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini, whichever you have
   access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, check the code again: who's
   right?

This is a self-check on your own understanding: a way to find where your mental model of
the code breaks down against a second reader.